# LAVKA PO1 Incremental Loader

Инкрементальная загрузка данных Лавки PO1 из SharePoint → `ecom_sandbox`.  
Три таблицы: `lavka_po1_metrics`, `lavka_po1_distr`, `lavka_po1_assort`.  
В каждой: `metric_version` (дата из имени файла) + `_source_file` (имя файла).

In [ ]:
import os, re, shutil
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
team_folder = notebook_path.split("/")[2]

import sys
sys.path.append(
    f"/Workspace/eperfectstore-prod/{team_folder}/notebooks/"
    f"eperfectstore-prod/e-com/COMMON_FUNCTIONS_AND_CONSTANTS_FOLDER/"
)
import common_functions_and_constants as CF

In [ ]:
SP_BASE = "https://pepsico.sharepoint.com/teams/RussiaSPO1CustomerCollaboration/"


def parse_date_from_filename(filename):
    """Ищет дату ddmmyyyy в имени файла → date или None."""
    m = re.search(r'(?<!\d)(\d{2})(\d{2})(\d{4})(?!\d)', os.path.basename(filename))
    if not m:
        return None
    d, mo, y = m.groups()
    try:
        return pd.to_datetime(f"{y}-{mo}-{d}").date()
    except Exception:
        return None


def excel_id_to_str(x):
    """Конвертирует Excel ID (float/scientific notation) в строку."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith(".0"):
        s = s[:-2]
    try:
        if "E+" in s.upper() or "E-" in s.upper():
            s = str(int(float(s)))
    except Exception:
        pass
    return s


def cleanup_and_download(sp_path, temp_path):
    """Очищает temp-папку и скачивает файлы из SharePoint."""
    if os.path.exists(temp_path):
        for name in os.listdir(temp_path):
            p = os.path.join(temp_path, name)
            if os.path.isfile(p) or os.path.islink(p):
                os.remove(p)
            elif os.path.isdir(p):
                shutil.rmtree(p)
    else:
        os.makedirs(temp_path, exist_ok=True)
    CF.copy_from_spo(SP_BASE, sp_path, temp_path)


def get_new_files(temp_path, target_table):
    """
    Список файлов для загрузки.
    Если таблица существует — только файлы с датой > max(metric_version).
    Иначе — все файлы.
    """
    all_files = sorted([
        f for f in os.listdir(temp_path)
        if os.path.isfile(os.path.join(temp_path, f)) and not f.startswith("~$")
    ])

    try:
        max_ver = spark.table(target_table).select(F.max("metric_version")).first()[0]
    except AnalysisException:
        max_ver = None

    if max_ver is None:
        return all_files

    return [
        f for f in all_files
        if (dt := parse_date_from_filename(f)) is not None
        and pd.to_datetime(dt) > pd.to_datetime(max_ver)
    ]


def write_incremental(pdf_list, target_table):
    """Concat pandas DFs → Spark DF → append в target_table."""
    if not pdf_list:
        print(f"{target_table}: нет новых файлов")
        return
    sdf = spark.createDataFrame(pd.concat(pdf_list, ignore_index=True))
    sdf.write.mode("append").saveAsTable(target_table)
    print(f"{target_table}: +{sdf.count()} rows")

## po1_metrics

In [ ]:
SP_PATH = "Shared Documents/General/E-COM/Клиенты/2.Лавка/DOS отчет/DOS REP/Метрики cpfr - CSV/"
TEMP   = "/dbfs/ecom/lavka/po1/metrics"
TARGET = "ecom_sandbox.lavka_po1_metrics"

cleanup_and_download(SP_PATH, TEMP)
files = get_new_files(TEMP, TARGET)

dfs = []
for fname in files:
    pdf = pd.read_csv(os.path.join(TEMP, fname))
    pdf.columns = [str(c).strip() for c in pdf.columns]

    pdf = pdf.rename(columns={
        "Date": "date", "City": "city", "Supplier": "supplier",
        "Item": "item", "Item_Name": "item_name",
        "Metric": "metric", "Value": "value",
    })

    pdf["date"]  = pd.to_datetime(pdf["date"], errors="coerce")
    pdf["value"] = pd.to_numeric(pdf["value"], errors="coerce")

    mv = parse_date_from_filename(fname)
    pdf["metric_version"] = pd.to_datetime(mv) if mv else pd.NaT
    pdf["_source_file"]   = fname
    dfs.append(pdf)

write_incremental(dfs, TARGET)

## po1_distr

In [ ]:
SP_PATH = "Shared Documents/General/E-COM/Клиенты/2.Лавка/Для КУБ Лавка/"
TEMP   = "/dbfs/ecom/lavka/po1/distr"
TARGET = "ecom_sandbox.lavka_po1_distr"

cleanup_and_download(SP_PATH, TEMP)
files = get_new_files(TEMP, TARGET)

dfs = []
for fname in files:
    sheets = pd.read_excel(os.path.join(TEMP, fname), sheet_name=None, header=0)
    mv = parse_date_from_filename(fname)

    for sheet_name, sheet_df in sheets.items():
        if sheet_df is None or sheet_df.empty:
            continue

        pdf = sheet_df.copy()
        pdf.columns = [str(c).strip() for c in pdf.columns]

        pdf = pdf.rename(columns={
            "City Name": "city_name", "Store ID": "store_id",
            "Store Name": "store_name", "Item ID": "item_id",
            "Item Name": "item_name",
        })
        for col in ["city_name", "store_id", "store_name", "item_id", "item_name"]:
            if col in pdf.columns:
                pdf[col] = pdf[col].astype(str)

        pdf["sheet_name"] = sheet_name

        # melt: колонки-даты → строки (date, value)
        # pandas может авто-парсить заголовки dd.mm.yyyy в Timestamp,
        # поэтому ловим оба варианта: "02.03.2026" и "2026-03-02 00:00:00"
        id_cols = ["city_name", "store_id", "store_name", "item_id", "item_name", "sheet_name"]
        date_cols = [
            c for c in pdf.columns
            if c not in id_cols and c != "Итого"
            and (re.fullmatch(r"\d{2}\.\d{2}\.\d{4}", str(c))
                 or re.fullmatch(r"\d{4}-\d{2}-\d{2}.*", str(c)))
        ]

        if date_cols:
            pdf = pdf.melt(
                id_vars=[c for c in id_cols if c in pdf.columns],
                value_vars=date_cols, var_name="date", value_name="value",
            )
            pdf["date"]  = pd.to_datetime(pdf["date"], dayfirst=True, errors="coerce")
            pdf["value"] = pd.to_numeric(pdf["value"], errors="coerce")

        pdf["metric_version"] = pd.to_datetime(mv) if mv else pd.NaT
        pdf["_source_file"]   = fname
        dfs.append(pdf)

write_incremental(dfs, TARGET)

## po1_dict (assort)

In [ ]:
SP_PATH = "Shared Documents/General/E-COM/Клиенты/2.Лавка/DICTIONARIES/"
TEMP    = "/dbfs/ecom/lavka/po1/assort"
TARGET  = "ecom_sandbox.lavka_po1_assort"
FILE    = "Assort Lavka.xlsx"

cleanup_and_download(SP_PATH, TEMP)

pdf = pd.read_excel(
    os.path.join(TEMP, FILE), sheet_name="Assort", header=0,
    converters={"SAP": excel_id_to_str, "GTIN": excel_id_to_str, "LAVKA": excel_id_to_str},
)
pdf.columns = [str(c).strip() for c in pdf.columns]

pdf = pdf.rename(columns={
    "Category": "category", "Brand": "brand", "subbrand": "subbrand",
    "Size": "size", "SAP": "sap", "GTIN": "gtin", "LAVKA": "lavka",
    "SKU": "sku", "MKU": "mku", "STATUS": "status", "GEO": "geo",
    "CPFR": "cpfr", "MKU ENG": "mku_eng", "Distribution": "distribution",
    "DC": "dc", "VOL": "vol",
})

pdf["vol"] = pd.to_numeric(pdf.get("vol"), errors="coerce")
for col in ["category", "brand", "subbrand", "size", "sap", "gtin", "lavka",
            "sku", "mku", "status", "geo", "cpfr", "mku_eng", "distribution", "dc"]:
    if col in pdf.columns:
        pdf[col] = pdf[col].astype(str)

pdf["_source_file"] = FILE

sdf = spark.createDataFrame(pdf)
sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET)
print(f"{TARGET}: overwritten, {sdf.count()} rows")